# extract_pdf_highlights.ipynb

In [1]:
from pdf_highlight_extractor.reader import extract_highlights
from icecream import ic
import pandas as pd
import sqlite3

In [2]:
# Variables for this run
marked_pdf = 'report_2026-02-24_15-11-32_printer.pdf'
csv_output = 'report_2026-02-24_15-11-32_printer.csv'
db_path = 'sam3_detections.sqlite3'

# Functions

In [3]:
def get_attributes_from_pdf(marked_pdf: str) -> pd.DataFrame:
    """
    Extracts highlights from a PDF and organizes them into a DataFrame containing attributes for each detection_id.
    """
    highlights = extract_highlights(marked_pdf)
    df = pd.DataFrame(highlights)
    df = df.drop(columns=['color'])  
    
    mylist = []
    mydict = {}
    for i, r in df.iterrows():
        if r['text'].startswith('detection_id'):
            if len(mydict) > 0:
                mylist.append(mydict)
            mydict = {'page': r['page'], 'detection_id': int(r['text'].replace('detection_id:', ''))}
        else:
            mydict[r['text']] = True
    mylist.append(mydict)
    
    df_attributes = pd.DataFrame(mylist)
    df_attributes = df_attributes.fillna(False)
    df_attributes.rename(columns={
        'accept': 'is_accepted',
        'healthy': 'is_healthy',
        'damaged_with_vcuts': 'is_damaged_with_vcuts',
        'damaged_without_vcuts': 'is_damaged_without_vcuts',
        'dead': 'is_dead',
        'reject': 'is_rejected',
        'crowd': 'is_crowded',
        'occluded': 'is_occluded',
        'other_problems': 'has_other_problems'
        }, 
        inplace=True)
    
    return df_attributes

In [4]:
def update_detections_table_with_attributes(db_path, df: pd.DataFrame):
    """
    Generates SQL UPDATE statements to update the detections table based on the attributes in the DataFrame.
    """
    column_names = df.columns.tolist()[2:]  # Exclude 'page' and 'detection_id'
    ic(column_names)
    for i, r in df.iterrows():
        for column_name in column_names:
            value = r[column_name]
            detection_id = r['detection_id']
            sql = f"UPDATE detections SET {column_name} = {value} WHERE detection_id = {detection_id};"
            ic(sql)
            try:
                with sqlite3.connect(db_path) as conn:
                    cursor = conn.cursor()
                    cursor.execute(sql)
                    # The transaction is automatically committed when the 'with' block exits
            except sqlite3.OperationalError as e:
                print(f"Database error: {e}")

# Main

In [5]:
df = get_attributes_from_pdf(marked_pdf)
update_detections_table_with_attributes(db_path, df)

with sqlite3.connect(db_path) as conn:
    df_from_db = pd.read_sql_query("SELECT * FROM detections", conn)

ic| column_names: ['is_accepted',
                   'is_damaged_with_vcuts',
                   'is_rejected',
                   'is_crowded',
                   'is_damaged_without_vcuts',
                   'is_occluded',
                   'is_dead']
ic| sql: 'UPDATE detections SET is_accepted = True WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_damaged_with_vcuts = True WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_rejected = False WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_crowded = False WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_damaged_without_vcuts = False WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_occluded = False WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_dead = False WHERE detection_id = 27;'
ic| sql: 'UPDATE detections SET is_accepted = True WHERE detection_id = 26;'
ic| sql: 'UPDATE detections SET is_damaged_with_vcuts = True WHERE detection_id = 26;'
ic| sql: 'UPDATE d

In [7]:
ic(df.columns)
ic(df_from_db.columns)
df_from_db

ic| df.columns: Index(['page', 'detection_id', 'is_accepted', 'is_damaged_with_vcuts',
                       'is_rejected', 'is_crowded', 'is_damaged_without_vcuts', 'is_occluded',
                       'is_dead'],
                      dtype='str')
ic| df_from_db.columns: Index(['detection_id', 'image_id', 'class_id', 'poly_wkt', 'poly_wkt_c',
                               'x_min', 'y_min', 'x_max', 'y_max', 'confidence', 'is_accepted',
                               'is_healthy', 'is_damaged_with_vcuts', 'is_damaged_without_vcuts',
                               'is_dead', 'is_rejected', 'is_crowded', 'is_occluded',
                               'has_other_problem'],
                              dtype='str')


,detection_id,image_id,class_id,poly_wkt,poly_wkt_c,x_min,y_min,x_max,y_max,confidence,is_accepted,is_healthy,is_damaged_with_vcuts,is_damaged_without_vcuts,is_dead,is_rejected,is_crowded,is_occluded,has_other_problem
0,1,1,0,"POLYGON ((1339 805, 1338 804, 1338 803, 1337 8...","POLYGON ((1339 275, 1338 276, 1338 277, 1337 2...",1342,709,1462,992,0.649902,0,0,0,0,0,1,1,0,0
1,2,1,0,"POLYGON ((437 934, 437 935, 438 936, 439 936, ...","POLYGON ((437 146, 437 145, 438 144, 439 144, ...",266,762,338,944,0.259521,0,0,0,0,0,1,0,1,0
2,3,1,0,"POLYGON ((538 795, 537 796, 535 796, 534 797, ...","POLYGON ((538 285, 537 284, 535 284, 534 283, ...",491,794,561,961,0.698730,1,0,0,1,0,0,0,0,0
3,4,1,0,"POLYGON ((902 863, 902 865, 901 866, 901 867, ...","POLYGON ((902 217, 902 215, 901 214, 901 213, ...",817,790,938,971,0.743164,1,0,1,0,0,1,0,1,0
4,5,1,0,"POLYGON ((906 838, 906 835, 906 838, 943 847, ...","POLYGON ((906 242, 906 245, 906 242, 943 233, ...",938,801,963,990,0.479980,1,0,0,0,1,0,0,0,0
5,6,1,0,"POLYGON ((1681 715, 1680 716, 1680 717, 1675 7...","POLYGON ((1681 365, 1680 364, 1680 363, 1675 3...",1592,715,1778,905,0.771484,0,0,0,0,0,1,1,0,0
6,7,1,0,"POLYGON ((553 936, 553 938, 554 939, 554 940, ...","POLYGON ((553 144, 553 142, 554 141, 554 140, ...",557,801,682,958,0.348389,0,0,0,0,0,1,0,1,0
7,8,1,0,"POLYGON ((1130 942, 1130 945, 1129 946, 1129 9...","POLYGON ((1130 138, 1130 135, 1129 134, 1129 1...",1046,801,1136,977,0.677734,0,0,0,0,0,1,1,0,0
8,9,1,0,"POLYGON ((1519 750, 1519 751, 1518 752, 1518 7...","POLYGON ((1519 330, 1519 329, 1518 328, 1518 3...",1483,748,1624,902,0.616211,0,0,0,0,0,1,1,0,0
9,10,1,0,"POLYGON ((1013 829, 1008 834, 1006 834, 1003 8...","POLYGON ((1013 251, 1008 246, 1006 246, 1003 2...",963,828,1055,985,0.763672,1,0,0,1,0,0,0,0,0
